# GPQA Diamond - ReAct Inference
## Evaluate PhD-level question answering using ReAct (Reasoning + Acting) method
- Loads GPQA Diamond dataset from HuggingFace (198 PhD-level questions)
- Subjects: Biology, Chemistry, Physics (multiple-choice)
- Uses ReAct reasoning for question-answering
- Extracts A/B/C/D answers from model reasoning
- Calculates exact match accuracy

In [32]:
# Setup - Change as needed for local environment
import os
import sys
from pathlib import Path

# For Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    BASE_PATH = '/content/drive/MyDrive/AGoT-ReAct/Math Performance'
except:
    IS_COLAB = False
    # For local: Use parent directory of current notebook
    BASE_PATH = str(Path(__file__).parent.parent) if '__file__' in globals() else str(Path.cwd().parent)
    # Fallback: Detect from workspace
    if not os.path.exists(BASE_PATH) or 'ReAct' not in BASE_PATH:
        BASE_PATH = r'f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance'

print(f"Environment: {'Colab' if IS_COLAB else 'Local'}")
print(f"Base path: {BASE_PATH}")
print(f"Base path exists: {os.path.exists(BASE_PATH)}")

Environment: Local
Base path: f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance
Base path exists: True


In [2]:
# Install dependencies from requirements.txt
!pip install -q -r requirements.txt

print("✓ All dependencies installed from requirements.txt")

✓ All dependencies installed from requirements.txt


In [33]:
import os
import json
import time
import re
from pathlib import Path
from getpass import getpass
from datetime import datetime
import pandas as pd
from tqdm import tqdm
import torch
from openai import OpenAI

# Setup paths
OUTPUT_DIR = Path(BASE_PATH) / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GPQA_OUTPUT_PATH = OUTPUT_DIR / 'gpqa_react_results.jsonl'
GPQA_METRICS_PATH = OUTPUT_DIR / 'gpqa_metrics.json'

# Configure OpenAI API
MODEL_NAME = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass('Enter your OpenAI API key: ')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

client = OpenAI(api_key=OPENAI_API_KEY)

print(f'Model: {MODEL_NAME}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Ready for GPQA Diamond evaluation')

Model: gpt-4o-mini
Output directory: f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance\outputs
Ready for GPQA Diamond evaluation


In [17]:
# Load GPQA Diamond dataset
print("Loading GPQA Diamond dataset from HuggingFace...")
print("Dataset: 198 PhD-level multiple-choice questions in Biology, Chemistry, Physics\n")
try:
    from datasets import load_dataset
    gpqa_dataset = load_dataset("fingertap/GPQA-Diamond", split="test")
    print(f"✓ Loaded {len(gpqa_dataset)} examples from GPQA Diamond (train split)")
except Exception as e:
    print(f"Train split error: {e}")
    gpqa_dataset = load_dataset("fingertap/GPQA-Diamond", split="test")
    print(f"✓ Loaded {len(gpqa_dataset)} examples from GPQA Diamond (test split)")

print(f"\nDataset features: {gpqa_dataset.column_names}")
print(f"\nFirst example:")
print(json.dumps({k: str(v)[:100] for k, v in gpqa_dataset[0].items()}, indent=2))

Loading GPQA Diamond dataset from HuggingFace...
Dataset: 198 PhD-level multiple-choice questions in Biology, Chemistry, Physics

✓ Loaded 198 examples from GPQA Diamond (train split)

Dataset features: ['question', 'answer']

First example:
{
  "question": "Among the following exoplanets, which one has the highest density?\n\na) An Earth-mass and Earth-radiu",
  "answer": "D"
}


## Demo: Test ReAct on Sample Question

In [19]:
# Demo: Test ReAct on a sample physics/math question
# IMPORTANT: GPQA format has options a/b/c/d and answer mapping A/B/C/D
# Example: If correct is "c) 5x massive" and mapping shows "D. c", answer is D
demo_question = """Among the following exoplanets, which one has the highest density?

a) An Earth-mass and Earth-radius planet.
b) A planet with 2 Earth masses and a density of approximately 5.5 g/cm^3.
c) A planet with the same composition as Earth but 5 times more massive than Earth.
d) A planet with the same composition as Earth but half the mass of Earth.

A. d
B. a
C. b
D. c"""

# Keep ORIGINAL format - DO NOT reformat a/b/c/d to A/B/C/D
# The question already has the correct structure with mapping
demo_formatted = demo_question

demo_example = {
    'index': 0,
    'question': demo_question,
    'formatted_question': demo_formatted,
    'correct_answer': 'D'  # Correct: c) 5x massive, and mapping shows D. c → Answer is D
}

print("="*70)
print("DEMO: GPQA-STYLE QUESTION - ReAct SOLVING")
print("="*70)
print("\nQuestion:")
print(demo_formatted)
print("\n" + "="*70)
print("Running ReAct solver...")
print("="*70)

# Note: This will only work after external_tools and react_solve_question are defined
# Run this cell after executing cells 7 and 9
try:
    demo_result = react_solve_question(demo_example, max_iterations=6, temperature=0.3)
    
    print("\n" + "="*70)
    print("REACT REASONING TRACE:")
    print("="*70)
    print(demo_result["react_trace"])
    
    print("\n" + "="*70)
    print("FINAL RESULT:")
    print("="*70)
    print(f"Model Answer: {demo_result['react_answer']}")
    print(f"Correct Answer: {demo_result['correct_answer']}")
    print(f"Result: {'✓ CORRECT' if demo_result['is_correct'] else '✗ INCORRECT'}")
    print("="*70)
    
except NameError as e:
    print(f"\n⚠️ Please run cells 7 and 9 first to define external_tools and react_solve_question")
    print(f"Error: {e}")

DEMO: GPQA-STYLE QUESTION - ReAct SOLVING

Question:
Among the following exoplanets, which one has the highest density?

a) An Earth-mass and Earth-radius planet.
b) A planet with 2 Earth masses and a density of approximately 5.5 g/cm^3.
c) A planet with the same composition as Earth but 5 times more massive than Earth.
d) A planet with the same composition as Earth but half the mass of Earth.

A. d
B. a
C. b
D. c

Running ReAct solver...

REACT REASONING TRACE:
Thought 1: 
Action 1: 1. The core question is asking which of the given exoplanets has the highest density.
2. The relevant concepts here involve the definition of density (mass/volume) and how it changes with different mass and radius configurations.
3. I know the formula for density and the relationship between mass, radius, and volume. I need to calculate the densities of each option to compare them.
4. I will calculate the density for each option systematically using the formula for density and the relationship between mass

## ReAct PhD-level Question Solver Implementation

In [24]:
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import quote_plus

class ExternalToolExecutor:
    """Executes external API calls for ReAct Actions"""
    
    def __init__(self):
        self.search_history = []
        
    def search_wikipedia(self, entity: str) -> str:
        """
        External Tool: Search Wikipedia for entity
        Returns: First few sentences or similar entities
        Uses Wikipedia API for reliable results
        """
        try:
            import urllib.parse
            
            # Use Wikipedia API for better reliability
            api_url = "https://en.wikipedia.org/w/api.php"
            params = {
                'action': 'query',
                'format': 'json',
                'titles': entity,
                'prop': 'extracts',
                'explaintext': True,
                'exintro': True,  # Get intro section only
                'redirects': 1  # Follow redirects
            }
            
            response = requests.get(api_url, params=params, timeout=10)
            data = response.json()
            
            # Extract page content
            pages = data.get('query', {}).get('pages', {})
            if pages:
                page_id = list(pages.keys())[0]
                page = pages[page_id]
                
                # Check if page exists
                if 'missing' in page:
                    return f"Wikipedia page for '{entity}' not found."
                
                extract = page.get('extract', '')
                if extract:
                    # Limit to ~200 words
                    words = extract.split()[:200]
                    result = ' '.join(words)
                    return result + ('...' if len(extract.split()) > 200 else '')
                else:
                    return f"No content found for '{entity}'."
            else:
                return f"Could not retrieve Wikipedia page for '{entity}'."
                    
        except Exception as e:
            return f"Wikipedia search failed: {str(e)[:80]}"
    
    def search_web(self, query: str) -> str:
        """
        External Tool: Simple web search using DuckDuckGo HTML
        Returns: Search snippets with reliable extraction
        """
        try:
            # Using DuckDuckGo as simple alternative (no API key needed)
            query_encoded = quote_plus(query)
            search_url = f"https://html.duckduckgo.com/html/?q={query_encoded}"
            
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
            response = requests.get(search_url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, features="html.parser")
            
            # Extract result snippets - try multiple selectors for robustness
            results = []
            
            # Try primary selector
            result_items = soup.find_all("div", {"class": "result"})
            for item in result_items[:3]:
                snippet = item.find("a", {"class": "result__snippet"})
                if snippet:
                    text = snippet.get_text().strip()
                    if text and len(text) > 10:
                        results.append(text)
            
            # Fallback: try other patterns
            if not results:
                paragraphs = soup.find_all("p", limit=3)
                for p in paragraphs:
                    text = p.get_text().strip()
                    if text and len(text) > 20:
                        results.append(text)
            
            if results:
                combined = " ".join(results[:3])
                # Limit to ~150 words
                words = combined.split()[:150]
                return ' '.join(words) + ('...' if len(combined.split()) > 150 else '')
            else:
                return f"Web search could not find readable results for '{query}'. Try a more specific search term."
                
        except Exception as e:
            return f"Web search failed for '{query}': {str(e)[:60]}"
    
    def lookup_in_text(self, keyword: str, context: str) -> str:
        """
        External Tool: Find keyword in previously retrieved text
        Returns: Sentence(s) containing keyword with context
        """
        try:
            if not context or len(context.strip()) == 0:
                return f"No context available to search for '{keyword}'."
            
            # Split by sentence
            sentences = context.replace('\n', ' ').split('.')
            matches = []
            
            for sentence in sentences:
                if keyword.lower() in sentence.lower():
                    clean_sent = sentence.strip()
                    if len(clean_sent) > 5:  # Only meaningful sentences
                        matches.append(clean_sent)
            
            if matches:
                # Return first 2 matching sentences
                result = '. '.join(matches[:2]) + '.'
                # Limit to ~100 words
                words = result.split()[:100]
                return ' '.join(words)
            else:
                return f"Keyword '{keyword}' not found in the retrieved text. Try a related term or search for new information."
        except Exception as e:
            return f"Lookup failed: {str(e)[:60]}"

# Initialize external tools
external_tools = ExternalToolExecutor()
print("External tool executor initialized (Wikipedia + Web Search)")

External tool executor initialized (Wikipedia + Web Search)


## ReAct Prompt and Inference

In [ ]:
import re
import time


def extract_final_answer(text: str) -> str:
    """Extract ONLY the answer letter A/B/C/D from text"""
    # Remove all whitespace and convert to uppercase
    text_clean = re.sub(r'\s+', '', text.upper())
    
    # Pattern 1: finish[A/B/C/D]
    match = re.search(r'FINISH\[([A-D])\]', text_clean)
    if match:
        return match.group(1)
    
    # Pattern 2: Final Answer: X
    match = re.search(r'FINALANSWER\s*:\s*([A-D])', text_clean)
    if match:
        return match.group(1)
    
    # Pattern 3: Answer: X or ANSWER: X
    match = re.search(r'ANSWER\s*:\s*([A-D])', text_clean)
    if match:
        return match.group(1)
    
    # Pattern 4: Standalone letter at end
    match = re.search(r'([A-D])', text[-50:].upper())
    if match:
        return match.group(1)
    
    return ""


def build_react_prompt(question: str) -> str:
    """Build ReAct prompt for PhD-level question answering with CoT-enhanced reasoning"""
    instruction = """Solve this PhD-level multiple-choice question using ReAct (Reasoning + Acting).
Subjects: Biology, Chemistry, Physics

You can use these external tools:
- search[entity]: Search Wikipedia for information about concepts, constants, definitions, or theories
- lookup[keyword]: Find specific information in previously retrieved text
- websearch[query]: Search the web for scientific concepts, formulas, or principles
- finish[answer]: Return final answer as SINGLE LETTER (A, B, C, or D)

Format (using Chain-of-Thought reasoning in each Thought step):
Thought: [Think step-by-step:
  1. What is the core question asking?
  2. What key concepts, theories, or formulas are relevant?
  3. What information do I already know vs. what do I need to search for?
  4. How should I approach solving this systematically?
  Reason through each step clearly before deciding on an action.]
Action: [tool_name[parameter]]
Observation: [tool result will be provided]

Repeat: Thought → Action → Observation until you have enough information and reasoning to confidently answer.

IMPORTANT: In each Thought, use Chain-of-Thought reasoning:
- Break down the problem into smaller parts
- Identify assumptions and validate them
- Show your mathematical/logical reasoning explicitly
- Compare options systematically
- Verify your conclusion before finishing

When you have the answer, use: Action: finish[X] where X is A, B, C, or D ONLY.

Question: """
    return instruction + question + "\n"


def parse_action(text: str) -> tuple:
    """
    Parse Action from LLM output
    Returns: (action_type, parameter) or (None, None)
    """
    # Match patterns: search[...], lookup[...], websearch[...], finish[...]
    patterns = [
        (r'search\[(.+?)\]', 'search'),
        (r'lookup\[(.+?)\]', 'lookup'),
        (r'websearch\[(.+?)\]', 'websearch'),
        (r'finish\[([A-D])\]', 'finish'),
    ]
    
    text_lower = text.lower()
    for pattern, action_type in patterns:
        match = re.search(pattern, text_lower, re.IGNORECASE | re.DOTALL)
        if match:
            param = match.group(1).strip()
            return (action_type, param)
    
    return (None, None)


def execute_external_tool(action_type: str, parameter: str, context: dict) -> str:
    """
    Execute external tool and return Observation
    """
    if action_type == 'search':
        return external_tools.search_wikipedia(parameter)
    elif action_type == 'websearch':
        return external_tools.search_web(parameter)
    elif action_type == 'lookup':
        prev_context = context.get('last_observation', '')
        return external_tools.lookup_in_text(parameter, prev_context)
    elif action_type == 'finish':
        return f"Answer: {parameter.upper()}"
    else:
        return "Invalid action."


def llm_call(prompt: str, stop_tokens: list = None, attempt: int = 1, temperature: float = 0.3) -> str:
    """Call model with retry logic"""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert at solving PhD-level science questions (Biology, Chemistry, Physics) using ReAct reasoning. "
                               "Generate Thought then Action. Wait for Observation before next step. "
                               "Be precise with domain knowledge, calculations, and logic. "
                               "When you have the final answer, use: finish[A] or finish[B] or finish[C] or finish[D]"
                },
                {"role": "user", "content": prompt},
            ],
            temperature=temperature,
            max_tokens=500,
            stop=stop_tokens,
        )
        text = response.choices[0].message.content or ""
        if not text.strip():
            raise ValueError("LLM returned no text")
        return text
    except Exception as e:
        if attempt < 3:
            print(f"  Attempt {attempt} failed: {e}, retrying...")
            time.sleep(2)
            return llm_call(prompt, stop_tokens, attempt + 1, temperature)
        else:
            raise


def fallback_solve(question: str) -> str:
    """Simple fallback solution if ReAct fails"""
    try:
        prompt = f"Answer this PhD-level science question with ONLY the letter (A, B, C, or D):\n{question}\nYour answer (just the letter):"
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are a science expert. Respond with ONLY a single letter: A, B, C, or D. Nothing else."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.1,
            max_tokens=5,
        )
        answer = (response.choices[0].message.content or "").strip().upper()
        # Extract only the letter
        answer = re.search(r'[A-D]', answer)
        return answer.group(0) if answer else ""
    except Exception:
        return ""


def react_solve_question(example: dict, max_iterations: int = 6, temperature: float = 0.3) -> dict:
    """
    ReAct question solver with EXTERNAL tool execution
    Implements true ReAct: Thought → Action[external_call] → Observation[external_result]
    For PhD-level science questions (Biology, Chemistry, Physics)
    Returns answer as ONLY A/B/C/D letter
    """
    question = example.get('formatted_question', '')
    correct_answer = example.get('correct_answer', '')
    
    prompt = build_react_prompt(question)
    steps = []
    final_answer = None
    context = {'last_observation': ''}
    
    try:
        for i in range(1, max_iterations + 1):
            # Generate Thought + Action
            thought_action = llm_call(
                prompt + f"Thought {i}:",
                stop_tokens=[f"\nObservation {i}:"],
                temperature=temperature
            )
            
            # Parse Thought and Action from LLM output
            # LLM output format: [thought content]\nAction {i}: [action]
            thought = ""
            action_text = ""
            
            # Split by "Action" to separate thought from action
            if f'action {i}:' in thought_action.lower():
                parts = re.split(rf'action\s+{i}\s*:', thought_action, flags=re.IGNORECASE)
                thought = parts[0].strip()
                action_text = parts[1].strip() if len(parts) > 1 else ""
            else:
                # No explicit Action label, try to find action pattern anyway
                thought = thought_action.strip()
                action_text = thought_action.strip()
            
            # Parse Action command to extract tool call
            action_type, parameter = parse_action(action_text)
            
            # Execute EXTERNAL tool
            if action_type == 'finish':
                final_answer = parameter.upper()
                observation = f"Answer: {final_answer}"
            elif action_type:
                # EXTERNAL API CALL
                observation = execute_external_tool(action_type, parameter, context)
                context['last_observation'] = observation
            else:
                # No valid action found
                observation = "No valid action detected. Please use search[], lookup[], websearch[], or finish[]."
            
            # Record step
            steps.append({
                "iteration": i,
                "thought": thought,
                "action": f"{action_type}[{parameter}]" if action_type else action_text,
                "observation": observation
            })
            
            # Update prompt with step
            step_str = f"Thought {i}: {thought}\nAction {i}: {action_text}\nObservation {i}: {observation}\n"
            prompt += step_str
            
            # Check if finished
            if action_type == 'finish' and final_answer in ['A', 'B', 'C', 'D']:
                break
        
        # Extract final answer
        if not final_answer or final_answer not in ['A', 'B', 'C', 'D']:
            final_answer = extract_final_answer(prompt)
        
        # Final fallback
        if not final_answer or final_answer not in ['A', 'B', 'C', 'D']:
            final_answer = fallback_solve(question)
            if not final_answer:
                final_answer = "?"
        
    except Exception as e:
        print(f"  ReAct failed: {e}, using fallback...")
        final_answer = fallback_solve(question)
        if not final_answer:
            final_answer = "?"
        steps.append({"thought": "Error occurred", "action": "fallback", "observation": str(e)[:100]})
    
    # Ensure final_answer is ONLY A/B/C/D
    if final_answer not in ['A', 'B', 'C', 'D']:
        final_answer = "?"
    
    # Build trace
    trace_lines = []
    for i, step in enumerate(steps, 1):
        trace_lines.append(f"Thought {i}: {step.get('thought', '')}")
        trace_lines.append(f"Action {i}: {step.get('action', '')}")
        trace_lines.append(f"Observation {i}: {step.get('observation', '')}")
    trace_lines.append(f"Final Answer: {final_answer}")
    trace = '\n'.join(trace_lines)
    
    return {
        "index": example.get("index"),
        "question": example.get('question', '')[:200],
        "correct_answer": correct_answer,
        "react_answer": final_answer,
        "is_correct": final_answer == correct_answer,
        "react_trace": trace,
        "steps": steps,
    }


In [41]:
# Format questions and run ReAct evaluation
print("Formatting GPQA Diamond questions...")
print("Note: GPQA includes PhD-level questions in Biology, Chemistry, and Physics\n")

# Format all questions first
# IMPORTANT: GPQA has a/b/c/d options with A/B/C/D answer mapping
# DO NOT reformat - keep original structure!
formatted_data = []
for idx, example in enumerate(gpqa_dataset):
    # GPQA Diamond structure: 'question' and 'answer' fields (lowercase)
    # question: contains full question with a/b/c/d options and A/B/C/D mapping
    # answer: contains the correct answer (single letter A/B/C/D)
    question = example.get('question', '')
    
    # Keep original format with a/b/c/d and A/B/C/D mapping
    formatted_q = question
    
    # Get correct answer - should already be A/B/C/D (single letter)
    correct_answer = example.get('answer', '').strip().upper()
    
    # Ensure it's a single letter
    if len(correct_answer) > 1:
        # Extract just the letter if format is like "A)" or "A."
        match = re.search(r'([A-D])', correct_answer.upper())
        if match:
            correct_answer = match.group(1)
    
    formatted_data.append({
        'index': idx,
        'question': question,
        'formatted_question': formatted_q,
        'correct_answer': correct_answer
    })

print(f"Formatted {len(formatted_data)} questions\n")

# Skip demo - go straight to batch evaluation
# Run evaluation on dataset with checkpoints
print("\n" + "="*60)
print("GPQA DIAMOND - REACT EVALUATION (WITH CHECKPOINTS)")
print("="*60)

# Checkpoint settings
BATCH_SIZE = 98  # Process 10 at a time
CHECKPOINT_PATH = OUTPUT_DIR / 'gpqa_react_checkpoint.json'
TRACES_PATH = OUTPUT_DIR / 'gpqa_react_detailed_traces.jsonl'  # Save detailed traces separately

# Load previous checkpoint if exists
checkpoint_data = {'evaluated_indices': set(), 'accumulated_results': []}
if CHECKPOINT_PATH.exists():
    try:
        with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
            saved_checkpoint = json.load(f)
            checkpoint_data['evaluated_indices'] = set(saved_checkpoint.get('evaluated_indices', []))
            checkpoint_data['accumulated_results'] = saved_checkpoint.get('accumulated_results', [])
        print(f"✓ Loaded checkpoint: {len(checkpoint_data['evaluated_indices'])} already evaluated")
    except Exception as e:
        print(f"⚠️ Could not load checkpoint: {e}")

# Find next batch to evaluate
all_indices = set(range(len(formatted_data)))
remaining_indices = sorted(all_indices - checkpoint_data['evaluated_indices'])

if remaining_indices:
    # Get next batch
    batch_indices = remaining_indices[:BATCH_SIZE]
    start_idx = batch_indices[0]
    end_idx = batch_indices[-1] + 1
    
    print(f"Evaluating batch: indices {start_idx}-{end_idx-1}")
    print(f"Progress: {len(checkpoint_data['evaluated_indices'])}/{len(formatted_data)} already done")
    print(f"Processing {len(batch_indices)} examples\n")
    
    results = []
    correct_count = 0
    
    for idx in tqdm(batch_indices, desc="ReAct Solving"):
        try:
            result = react_solve_question(formatted_data[idx], max_iterations=6, temperature=0.3)
            results.append(result)
            
            if result['is_correct']:
                correct_count += 1
            
            # Save to main results JSONL (with trace included)
            with open(GPQA_OUTPUT_PATH, 'a', encoding='utf-8') as f:
                json.dump({
                    "index": result['index'],
                    "question": result['question'],
                    "answer": result['react_answer'],
                    "correct_answer": result['correct_answer'],
                    "is_correct": result['is_correct'],
                    "react_trace": result['react_trace'],  # Full trace
                    "timestamp": datetime.now().isoformat()
                }, f, ensure_ascii=False)
                f.write("\n")
            
            # Save detailed trace separately for easier review
            with open(TRACES_PATH, 'a', encoding='utf-8') as f:
                json.dump({
                    "index": result['index'],
                    "question": result['question'][:150],
                    "correct_answer": result['correct_answer'],
                    "model_answer": result['react_answer'],
                    "is_correct": result['is_correct'],
                    "react_trace": result['react_trace'],
                    "steps": result['steps'],
                    "timestamp": datetime.now().isoformat()
                }, f, ensure_ascii=False)
                f.write("\n")
            
            # Update checkpoint after each question
            checkpoint_data['evaluated_indices'].add(idx)
            checkpoint_data['accumulated_results'].extend([result])
            
        except Exception as e:
            print(f"  ✗ Error on example {idx}: {str(e)[:60]}")
            continue
    
    # Save checkpoint
    with open(CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
        json.dump({
            'evaluated_indices': sorted(list(checkpoint_data['evaluated_indices'])),
            'accumulated_results': checkpoint_data['accumulated_results'],
            'timestamp': datetime.now().isoformat()
        }, f, ensure_ascii=False, indent=2)
    
    batch_accuracy = correct_count / len(batch_indices) * 100 if len(batch_indices) > 0 else 0
    
    print("\n" + "="*60)
    print("BATCH COMPLETE")
    print("="*60)
    print(f"Batch Accuracy: {batch_accuracy:.2f}%")
    print(f"Batch Correct: {correct_count}/{len(batch_indices)}")
    print(f"Total Evaluated So Far: {len(checkpoint_data['evaluated_indices'])}/{len(formatted_data)}")
    print(f"Checkpoint saved to: {CHECKPOINT_PATH}")
    print(f"Detailed traces saved to: {TRACES_PATH}")
    print("="*60)
    
else:
    print("✓ All 198 examples already evaluated!")
    print(f"Total Evaluated: {len(checkpoint_data['evaluated_indices'])}/198")
    results = checkpoint_data['accumulated_results']

Formatting GPQA Diamond questions...
Note: GPQA includes PhD-level questions in Biology, Chemistry, and Physics

Formatted 198 questions


GPQA DIAMOND - REACT EVALUATION (WITH CHECKPOINTS)
✓ Loaded checkpoint: 100 already evaluated
Evaluating batch: indices 100-197
Progress: 100/198 already done
Processing 98 examples



ReAct Solving: 100%|██████████| 98/98 [56:03<00:00, 34.32s/it]


BATCH COMPLETE
Batch Accuracy: 37.76%
Batch Correct: 37/98
Total Evaluated So Far: 198/198
Checkpoint saved to: f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance\outputs\gpqa_react_checkpoint.json
Detailed traces saved to: f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance\outputs\gpqa_react_detailed_traces.jsonl


In [42]:
# Analysis and Summary
# Calculate metrics from ALL results (both current batch and accumulated)
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("DETAILED ANALYSIS")
print("="*60)

correct_results = results_df[results_df['is_correct'] == True]
incorrect_results = results_df[results_df['is_correct'] == False]

# Calculate from actual results
total_evaluated = len(results_df)
batch_correct = len(correct_results)
batch_accuracy = batch_correct / total_evaluated * 100 if total_evaluated > 0 else 0

print(f"\n✓ Correct: {batch_correct} ({batch_correct/total_evaluated*100:.1f}%)")
print(f"✗ Incorrect: {len(incorrect_results)} ({len(incorrect_results)/total_evaluated*100:.1f}%)")

if len(incorrect_results) > 0:
    print("\nSample Incorrect Predictions (Answer format: A/B/C/D):")
    for idx, row in incorrect_results.head(3).iterrows():
        print(f"\n  Question: {row['question'][:80]}...")
        print(f"  Model Output: {row['react_answer']} | Correct: {row['correct_answer']}")

# Calculate CUMULATIVE metrics from checkpoint (all batches so far)
CUMULATIVE_RESULTS_PATH = OUTPUT_DIR / 'gpqa_react_cumulative_metrics.json'
all_evaluated_so_far = len(checkpoint_data['evaluated_indices'])

# Try to load previous cumulative metrics
cumulative_stats = {
    'total_all_batches': all_evaluated_so_far,
    'correct_all_batches': 0,
    'batches_completed': 0
}

if CUMULATIVE_RESULTS_PATH.exists():
    try:
        with open(CUMULATIVE_RESULTS_PATH, 'r', encoding='utf-8') as f:
            cumulative_stats = json.load(f)
    except:
        pass

# Update cumulative stats
cumulative_stats['total_all_batches'] = all_evaluated_so_far
cumulative_stats['correct_all_batches'] += batch_correct
cumulative_stats['batches_completed'] = (all_evaluated_so_far // BATCH_SIZE) + (1 if all_evaluated_so_far % BATCH_SIZE else 0)
cumulative_stats['last_batch_accuracy'] = batch_accuracy
cumulative_stats['last_updated'] = datetime.now().isoformat()

# Calculate cumulative accuracy
cumulative_accuracy = (cumulative_stats['correct_all_batches'] / all_evaluated_so_far * 100) if all_evaluated_so_far > 0 else 0

# Save cumulative metrics
with open(CUMULATIVE_RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(cumulative_stats, f, ensure_ascii=False, indent=2)

# Save batch metrics
metrics_summary = {
    "dataset": "GPQA-Diamond",
    "model": MODEL_NAME,
    "method": "ReAct",
    "current_batch": {
        "batch_size": len(batch_indices) if remaining_indices else 0,
        "correct_predictions": batch_correct,
        "total_evaluated": total_evaluated,
        "batch_accuracy": f"{batch_accuracy:.2f}%"
    },
    "cumulative": {
        "total_evaluated_all_batches": all_evaluated_so_far,
        "correct_all_batches": cumulative_stats['correct_all_batches'],
        "cumulative_accuracy": f"{cumulative_accuracy:.2f}%",
        "batches_completed": cumulative_stats['batches_completed'],
        "remaining_to_evaluate": len(formatted_data) - all_evaluated_so_far
    },
    "answer_format": "A/B/C/D only (single letter)",
    "checkpoint_path": str(CHECKPOINT_PATH),
    "traces_path": str(TRACES_PATH),
    "timestamp": datetime.now().isoformat(),
    "output_file": str(GPQA_OUTPUT_PATH)
}

with open(GPQA_METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

print("\n" + "="*60)
print("METRICS SUMMARY")
print("="*60)
print(json.dumps(metrics_summary, indent=2, ensure_ascii=False))
print("\n" + "="*60)
print("FILES SAVED:")
print("="*60)
print(f"✓ Results (with traces): {GPQA_OUTPUT_PATH}")
print(f"✓ Detailed traces: {TRACES_PATH}")
print(f"✓ Batch metrics: {GPQA_METRICS_PATH}")
print(f"✓ Cumulative metrics: {CUMULATIVE_RESULTS_PATH}")
print(f"✓ Checkpoint: {CHECKPOINT_PATH}")
print("="*60)


DETAILED ANALYSIS

✓ Correct: 37 (37.8%)
✗ Incorrect: 61 (62.2%)

Sample Incorrect Predictions (Answer format: A/B/C/D):

  Question: 7-(tert-butoxy)bicyclo[2.2.1]hepta-2,5-diene is combined with 2 equivalents of 5...
  Model Output: A | Correct: B

  Question: The Cope rearrangement is a chemical reaction where a 1,5-diene molecule undergo...
  Model Output: A | Correct: D

  Question: Identify the correct sequence of reagents for the synthesis of [1,1'-bi(cyclopen...
  Model Output: A | Correct: B

METRICS SUMMARY
{
  "dataset": "GPQA-Diamond",
  "model": "gpt-4o-mini",
  "method": "ReAct",
  "current_batch": {
    "batch_size": 98,
    "correct_predictions": 37,
    "total_evaluated": 98,
    "batch_accuracy": "37.76%"
  },
  "cumulative": {
    "total_evaluated_all_batches": 198,
    "correct_all_batches": 43,
    "cumulative_accuracy": "21.72%",
    "batches_completed": 3,
    "remaining_to_evaluate": 0
  },
  "answer_format": "A/B/C/D only (single letter)",
  "checkpoint_path": 